# Basic Polling Consumer — explained for a total Kafka beginner

This notebook assumes only one thing: you have seen that a **producer sends a message to Kafka**. Nothing else is assumed.

We will build one consumer slowly. We will not hide simple ideas behind words such as *deserialization*, *offset management*, or *at-least-once*. When a technical term is useful, we will first explain the ordinary idea and then give it a name.

## What this notebook deliberately does not contain

- No unit tests.
- No fake consumer.
- No large reusable consumer class.
- No concurrency, rebalance callbacks, retry framework, or dead-letter queue implementation.
- No claim that one short notebook solves every production failure.

Those subjects matter later. First, the basic consumer must make sense.

## The complete journey

```text
Stage 1: prepare settings
        |
        v
Stage 2: connect a consumer and subscribe to one topic
        |
        v
Stage 3: call poll() and understand its three possible answers
        |
        v
Stage 4: turn message bytes into a Python dictionary
        |
        v
Stage 5: perform the business work
        |
        v
Stage 6: commit only after that work succeeds
        |
        v
Stage 7: close the consumer even when something fails
```

That diagram is the entire notebook. Every later section only opens one box and explains it.


## Before code: the ten words you need

Read this table from top to bottom. Each definition uses ordinary language first.

| Kafka word | Plain-English meaning | Concrete example in this notebook |
|---|---|---|
| **Broker** | A running Kafka server that stores messages. | The local server at `localhost:9092`. |
| **Topic** | A named stream or category of messages. | `beginner_order_events` contains order events. |
| **Record / message** | One item stored in a topic. The two words are commonly used for the same thing here. | One `order.created` event. |
| **Producer** | Code that sends a record to Kafka. | The order service. |
| **Consumer** | Code that asks Kafka for records. | Our analytics program. |
| **Partition** | One numbered lane inside a topic. Each lane has its own order. | Partition `0`, partition `1`, and so on. |
| **Offset** | A record's position number inside one partition. | Offset `8` means the ninth stored position in that lane. |
| **Consumer group** | A named team of consumers that shares the work and remembers progress. | `beginner-order-analytics-v1`. |
| **Poll** | Ask the Kafka client, “Do you have the next record for me?” | `consumer.poll(1.0)`. |
| **Commit** | Save the group's next reading position in Kafka. | After successfully processing offset `8`, save position `9`. |

Do not memorize the table yet. The code will make each word concrete.


## One picture of producer and consumer

```text
ORDER SERVICE                                             ANALYTICS PROGRAM
(producer)                                                    (consumer)
    |                                                             ^
    | sends an order event                                        | poll asks for work
    v                                                             |
+--------------------------------------------------------------------------+
| Kafka broker                                                             |
|                                                                          |
| Topic: beginner_order_events                                             |
|                                                                          |
| Partition 0: [offset 0] [offset 1] [offset 2] ...                        |
| Partition 1: [offset 0] [offset 1] ...                                   |
+--------------------------------------------------------------------------+
```

Kafka keeps the records. The consumer does not remove a record when it reads it. Instead, the consumer group's saved position tells Kafka where that group should continue next time.

Think of the record as a page in a book and the committed offset as a bookmark. Reading a page does not tear it out of the book.


---
# Stage 1.0 — Prepare the notebook

## Stage 1.1 — Install and start the local tools

Install the Python client once:

```bash
python -m pip install "confluent-kafka>=2.3"
```

Start this repository's small learning broker from the repository root:

```bash
docker compose -f notebook/lab/docker-compose.yml up -d
```

The learning broker exposes Kafka at `localhost:9092`. This is a local-development setup, not a production cluster.

Jupyter remembers names created by earlier cells. Run the notebook from top to bottom. If you change an early cell, restart the kernel and run all cells again so old values do not confuse the result.


In [8]:
# Stage 1.2 — Import only the tools used by this lesson.
from __future__ import annotations

# Python standard-library imports: JSON conversion and environment settings.
import json
import os
from typing import Any

# Kafka is optional while reading, so a missing package does not break the lesson.
try:
    from confluent_kafka import Consumer, KafkaException, Message, Producer

    KAFKA_LIBRARY_AVAILABLE = True
except ImportError:
    Consumer = KafkaException = Message = Producer = None
    KAFKA_LIBRARY_AVAILABLE = False


### Code explanation — every meaningful line in Stage 1.2

Blank lines and closing brackets only make the code readable; they do not perform an action.

| Code | Meaning |
|---|---|
| `# Stage 1.2 ...` | A navigation marker. It tells you where this code belongs in the larger flow. |
| `from __future__ import annotations` | Keeps type hints from being evaluated immediately. That lets the notebook be readable even when the Kafka package is not installed yet. |
| `import json` | Loads Python's built-in JSON converter. We need it because Kafka gives us bytes, while our application wants a dictionary. |
| `import os` | Loads environment-variable support. An environment variable lets another machine change the broker address without editing this notebook. |
| `from typing import Any` | Imports a type meaning “the JSON value may have different shapes.” We use it only at the JSON boundary; after validation, the code checks the shape it needs. |
| `try:` | Starts an attempt that may fail safely. |
| `from confluent_kafka import ...` | Imports the real Kafka consumer, Kafka error wrapper, message type, and a tiny producer used only to seed the live exercise. |
| `KAFKA_LIBRARY_AVAILABLE = True` | Records that the import succeeded. |
| `except ImportError:` | Runs only when Python cannot find `confluent-kafka`. |
| `Consumer = ... = None` | Creates the expected names without pretending the real Kafka classes exist. Live functions check this before using them. |
| `KAFKA_LIBRARY_AVAILABLE = False` | Records that live Kafka code must not run yet. |

Why use a guarded import? You can read and understand the notebook before installing anything. When live execution is requested, the functions fail with a direct installation instruction.


In [9]:
# Stage 1.3 — Declare settings in one visible place.
BOOTSTRAP_SERVERS: str = os.getenv("KAFKA_BOOTSTRAP_SERVERS", "localhost:9092")
ORDER_TOPIC: str = "beginner_order_events"
ORDER_CONSUMER_GROUP: str = "beginner-order-analytics-v1"
RUN_LIVE_KAFKA: bool = False

# The values have different types, so Any is limited to this library configuration boundary.
CONSUMER_CONFIG: dict[str, Any] = {
    "bootstrap.servers": BOOTSTRAP_SERVERS,
    "group.id": ORDER_CONSUMER_GROUP,
    "auto.offset.reset": "earliest",
    "enable.auto.commit": False,
    "enable.auto.offset.store": False,
}

print("Kafka library installed:", KAFKA_LIBRARY_AVAILABLE)
print("Broker address:", BOOTSTRAP_SERVERS)
print("Topic:", ORDER_TOPIC)
print("Consumer group:", ORDER_CONSUMER_GROUP)
print("Live Kafka enabled:", RUN_LIVE_KAFKA)


Kafka library installed: True
Broker address: localhost:9092
Topic: beginner_order_events
Consumer group: beginner-order-analytics-v1
Live Kafka enabled: False


### Code explanation — every meaningful line in Stage 1.3

| Code | Meaning | When would production change it? |
|---|---|---|
| `BOOTSTRAP_SERVERS = os.getenv(...)` | Use `KAFKA_BOOTSTRAP_SERVERS` when it exists; otherwise use the local broker. “Bootstrap” only means the first address used to discover the Kafka cluster. | Use the real cluster endpoints and add authentication/TLS settings. |
| `ORDER_TOPIC = ...` | The exact topic name the producer and consumer must agree on. | Teams normally version and govern topic names. |
| `ORDER_CONSUMER_GROUP = ...` | The name of the team whose saved reading position we use. | Keep it stable for one application; unrelated applications need different group IDs. |
| `RUN_LIVE_KAFKA = False` | Safety switch. Reading cells cannot accidentally make a network connection. | A deployed service does not normally need this notebook switch. |
| `CONSUMER_CONFIG = {` | Begins the configuration dictionary passed to the Kafka client. `Any` is documented here because this third-party dictionary intentionally mixes strings and booleans. |
| `"bootstrap.servers": ...` | Tells the client how to find Kafka. It does not select a topic. |
| `"group.id": ...` | Tells Kafka which group's progress and partition ownership this consumer uses. |
| `"auto.offset.reset": "earliest"` | If this group has no usable saved position, begin at the oldest record Kafka still retains. It does **not** override an existing committed position. |
| `"enable.auto.commit": False` | Stops the client from periodically saving progress automatically. Our code will commit only after business work succeeds. |
| `"enable.auto.offset.store": False` | Stops the client from automatically marking a fetched record as ready for a later commit. This keeps “received” separate from “processed.” |
| Each `print(...)` | Shows the effective setting so a beginner can see exactly what the next cells will use. |

The two `False` values are intentional. They make progress handling more explicit and easier to reason about. They do not magically create exactly-once processing; we explain that limitation later.


---
# Stage 2.0 — See what Kafka actually stores

A producer usually starts with a Python dictionary. Kafka does not store that live Python object. The producer converts it to bytes before sending it.


In [10]:
# Stage 2.1 — Build one predictable order event.
SAMPLE_ORDER_EVENT: dict[str, Any] = {
    "event_id": "evt-1001",
    "event_type": "order.created",
    "schema_version": 1,
    "data": {
        "order_id": "order-1001",
        "customer_id": "customer-7",
        "total": 1499.0,
    },
}

# Stage 2.2 — Perform the same conversion a JSON producer performs.
json_text: str = json.dumps(SAMPLE_ORDER_EVENT)
wire_bytes: bytes = json_text.encode("utf-8")

print("Python value type:", type(SAMPLE_ORDER_EVENT).__name__)
print("JSON text type:", type(json_text).__name__)
print("Kafka value type:", type(wire_bytes).__name__)
print("Kafka bytes:", wire_bytes)


Python value type: dict
JSON text type: str
Kafka value type: bytes
Kafka bytes: b'{"event_id": "evt-1001", "event_type": "order.created", "schema_version": 1, "data": {"order_id": "order-1001", "customer_id": "customer-7", "total": 1499.0}}'


### Code explanation — every meaningful line in Stage 2

| Code | Meaning |
|---|---|
| `SAMPLE_ORDER_EVENT: dict[str, Any] = {` | Creates a Python dictionary. The type hint says its keys are strings and its values may have several JSON-compatible types. |
| `"event_id": "evt-1001"` | A unique event identity. This is business identity, not a Kafka offset. |
| `"event_type": "order.created"` | Says what happened. It is ordinary event data, not a Kafka command. |
| `"schema_version": 1` | Says which shape of this event the producer used. Consumers can reject versions they do not understand. |
| `"data": {` | Starts the part containing order-specific information. The surrounding fields form an “envelope”: a consistent wrapper around business data. |
| `order_id`, `customer_id`, `total` | The actual order fields used by our example business handler. |
| `json.dumps(SAMPLE_ORDER_EVENT)` | **Serializes** the dictionary: converts it into JSON text. “Serialize” simply means convert an in-memory value into a form that can be stored or sent. |
| `.encode("utf-8")` | Converts text into bytes. UTF-8 is the agreed rule for turning characters into byte numbers. |
| The four `print(...)` lines | Prove the three forms are different: Python dictionary → JSON string → bytes. `type(...).__name__` prints the simple type name. |

The consumer performs these conversions in reverse:

```text
Kafka bytes  --decode UTF-8-->  JSON text  --parse JSON-->  Python dictionary
```


## Stage 2.3 — Partition, offset, and key are different things

Suppose Kafka stores this:

```text
Topic: beginner_order_events

Partition 0:  offset 0     offset 1     offset 2
              order-10     order-12     order-15

Partition 1:  offset 0     offset 1
              order-11     order-14
```

- The **partition** is the lane number.
- The **offset** is a position inside that lane. Both partitions can have offset `0`; those are different records.
- The **message key** might be `order-10`. Producers commonly use a key to keep related records in the same partition.
- Kafka guarantees ordering inside one partition, not one total order across every partition.

A full Kafka location therefore needs three pieces: `topic + partition + offset`.


---
# Stage 3.0 — Convert untrusted bytes into a usable order event

“Deserialization” sounds grand, but it only means converting stored bytes back into an application value.

We also validate the result. Validation means refusing data whose shape our business code does not understand. Kafka transports bytes reliably; Kafka does not know whether your JSON contains an `order_id`.


In [11]:
class InvalidOrderEvent(ValueError):
    """Report that Kafka data cannot be used as this lesson's order event.

    Input boundary: raised for null, non-byte, non-JSON, or unsupported event data.
    Output contract: callers receive a clear failure instead of a partial event.
    Failure behavior: the polling loop stops before committing the bad record.
    """


def decode_order_event(raw_value: bytes | None) -> dict[str, Any]:
    """Decode and validate one Kafka value.

    Args:
        raw_value: Bytes returned by ``message.value()``, or ``None``.

    Returns:
        A dictionary containing one supported ``order.created`` event.

    Raises:
        InvalidOrderEvent: The value is missing, malformed, or unsupported.

    Production tradeoff:
        Stopping protects us from silently skipping unknown data, but one bad record
        can block this simple consumer until an operator fixes or quarantines it.
    """
    # Stage 3.1 — Check the raw Kafka value before decoding it.
    if raw_value is None:
        raise InvalidOrderEvent("The Kafka record has no value")
    if not isinstance(raw_value, bytes):
        raise InvalidOrderEvent("The Kafka value must be bytes")

    # Stage 3.2 — Convert bytes to text, then text to a Python value.
    try:
        json_text = raw_value.decode("utf-8")
        event = json.loads(json_text)
    except (UnicodeDecodeError, json.JSONDecodeError) as error:
        raise InvalidOrderEvent("The Kafka value must be valid UTF-8 JSON") from error

    # Stage 3.3 — Check the minimum business shape used by this lesson.
    if not isinstance(event, dict):
        raise InvalidOrderEvent("The JSON value must be an object")
    if event.get("event_type") != "order.created":
        raise InvalidOrderEvent("Only event_type='order.created' is supported")
    if event.get("schema_version") != 1:
        raise InvalidOrderEvent("Only schema_version=1 is supported")

    order_data = event.get("data")
    if not isinstance(order_data, dict):
        raise InvalidOrderEvent("The event must contain a data object")
    if not isinstance(order_data.get("order_id"), str) or not order_data["order_id"].strip():
        raise InvalidOrderEvent("data.order_id must be a non-empty string")

    # Stage 3.4 — Return only after all required checks pass.
    return event


decoded_order = decode_order_event(wire_bytes)
print("Decoded order ID:", decoded_order["data"]["order_id"])


Decoded order ID: order-1001


### Code explanation — every meaningful line in Stage 3

| Code | Meaning |
|---|---|
| `class InvalidOrderEvent(ValueError):` | Creates a specific error name. Catching this later is clearer than treating bad data as an unrelated network failure. It inherits normal `ValueError` behavior. |
| The class docstring | States why the class exists, what data triggers it, and the safety policy: do not commit an unusable record. |
| `def decode_order_event(...) -> dict[str, Any]:` | Defines a function that accepts bytes or `None` and promises a dictionary on success. `|` means “or.” Defining the function does not run it yet. |
| The function docstring | Documents the input, returned value, failures, and production tradeoff. |
| `if raw_value is None:` | Checks for a missing value. Kafka permits null values, but this order event contract does not. |
| `raise InvalidOrderEvent(...)` | Stops this function immediately and reports why. No dictionary is returned. |
| `if not isinstance(raw_value, bytes):` | Rejects accidental strings or other Python values. The live client normally returns bytes, but the boundary is explicit. |
| `try:` | Begins conversion operations that may fail for bad input. |
| `raw_value.decode("utf-8")` | Converts bytes into text using UTF-8. Invalid byte sequences raise `UnicodeDecodeError`. |
| `json.loads(json_text)` | Parses JSON text into a Python value. Broken JSON raises `JSONDecodeError`. |
| `except (...) as error:` | Catches either conversion error and gives the original exception the name `error`. |
| `raise ... from error` | Raises our beginner-friendly error while preserving the original technical cause for debugging. |
| `if not isinstance(event, dict):` | Rejects valid JSON values such as a list or string; this application requires an object/dictionary. |
| `event.get("event_type") != ...` | Reads the field safely. A missing field becomes `None` and is rejected. |
| `event.get("schema_version") != 1` | Refuses a data shape this code was not written to understand. |
| `order_data = event.get("data")` | Gives the nested business payload a readable name. |
| `if not isinstance(order_data, dict):` | Ensures later dictionary access is safe. |
| The `order_id` condition | Rejects a missing value, a non-string value, or a string containing only spaces. `or` means any failed requirement is enough to reject it. |
| `return event` | Finishes successfully only after every check has passed. |
| `decoded_order = decode_order_event(wire_bytes)` | Actually calls the function with our sample bytes. |
| The final `print(...)` | Reads the nested `order_id` to prove that the result is now usable application data. |


---
# Stage 4.0 — Keep business work separate from Kafka work

Kafka work is “poll a message and save progress.” Business work is “do something useful with the order.” Keeping them in separate functions makes failures easier to understand.

This lesson only appends an order ID to an in-memory list. A real handler might write an analytics row to a database. The list disappears when the notebook kernel stops, so it is demonstration state, not durable storage.


In [12]:
PROCESSED_ORDER_IDS: list[str] = []


def handle_order_created(event: dict[str, Any]) -> None:
    """Perform this lesson's business action for one validated event.

    Args:
        event: A dictionary already accepted by ``decode_order_event``.

    Returns:
        Nothing. Success means the function returns without raising an exception.

    Raises:
        KeyError: Required validated fields are unexpectedly absent.

    Production tradeoff:
        The in-memory list is intentionally simple and not durable. A real handler
        should make repeat execution safe before writing to an external system.
    """
    # Stage 4.1 — Extract the business identifier from the validated event.
    order_data = event["data"]
    order_id = order_data["order_id"]

    # Stage 4.2 — Perform the demonstration business action.
    PROCESSED_ORDER_IDS.append(order_id)
    print(f"Business work finished for order {order_id}")


handle_order_created(decoded_order)
print("Processed order IDs:", PROCESSED_ORDER_IDS)


Business work finished for order order-1001
Processed order IDs: ['order-1001']


### Code explanation — every meaningful line in Stage 4

| Code | Meaning |
|---|---|
| `PROCESSED_ORDER_IDS: list[str] = []` | Creates an empty list of strings. It stands in for a database so we can see the business result. |
| `def handle_order_created(...) -> None:` | Defines the business handler. `None` means it reports success by returning normally rather than returning data. |
| The function docstring | Describes its validated input, success contract, possible failure, and why an in-memory list is not production storage. |
| `order_data = event["data"]` | Enters the event's business payload. Square brackets are appropriate because Stage 3 already required this field. |
| `order_id = order_data["order_id"]` | Extracts the ID that the business action needs. |
| `PROCESSED_ORDER_IDS.append(order_id)` | Performs our demonstration action by adding the ID to the end of the list. |
| `print(f"...")` | Shows successful handling. The `f` allows `{order_id}` to be inserted into the text. |
| `handle_order_created(decoded_order)` | Calls the handler with the event decoded in Stage 3. |
| The final `print(...)` | Displays all demonstration work performed in this notebook kernel. |

The important boundary is this: if the handler raises an exception, the polling loop must not commit that record.


---
# Stage 5.0 — Understand `subscribe()` and `poll()` before using them

`subscribe([ORDER_TOPIC])` means: “Join my configured consumer group and ask Kafka to assign me work from this topic.” Assignment is not necessarily complete on that exact line; polling drives the group activity.

One call to `poll(1.0)` can produce three kinds of answer:

```text
                         consumer.poll(1.0)
                                  |
                 +----------------+----------------+
                 |                |                |
               None        message.error()    normal message
                 |                |                |
       nothing arrived in     Kafka/client     bytes + metadata
       this one-second wait       problem            |
                 |                |                  v
           ask again later   stop and report    decode and process
```

`None` is not proof that Kafka is broken and not proof that the topic will always be empty. It only says this particular wait ended without a record.

A normal message contains both data and location metadata:

| Expression | Answer |
|---|---|
| `message.value()` | Raw bytes (or `None`) |
| `message.key()` | Optional producer-supplied key bytes |
| `message.topic()` | Source topic name |
| `message.partition()` | Source lane number |
| `message.offset()` | Position in that lane |
| `message.error()` | Error information, or no error for a normal record |


In [13]:
def create_order_consumer() -> Consumer:
    """Create the real Kafka client using this notebook's visible configuration.

    Args:
        None. The function uses ``CONSUMER_CONFIG`` from Stage 1.

    Returns:
        A new, not-yet-subscribed ``confluent_kafka.Consumer``.

    Raises:
        RuntimeError: The Kafka Python package is not installed.
        KafkaException: The client rejects its configuration.

    Production tradeoff:
        Central configuration is easy to inspect. Real deployments should obtain
        broker addresses and credentials from managed configuration, not source code.
    """
    # Stage 5.1 — Refuse a vague error when the optional package is absent.
    if not KAFKA_LIBRARY_AVAILABLE or Consumer is None:
        raise RuntimeError("Install confluent-kafka before running the live lab")

    # Stage 5.2 — Allocate the client; this does not process a record.
    return Consumer(CONSUMER_CONFIG)


def commit_processed_message(consumer: Consumer, message: Message) -> None:
    """Save progress for exactly one successfully processed message.

    Args:
        consumer: The active Kafka consumer that returned the message.
        message: The normal record whose business handler completed successfully.

    Returns:
        Nothing after Kafka reports a successful synchronous commit.

    Raises:
        RuntimeError: Kafka returns no committed position or reports a commit error.
        KafkaException: The commit request itself fails.

    Production tradeoff:
        Waiting for every commit is easy to reason about but slower than batching.
        Faster commit strategies require more careful failure accounting.
    """
    # Stage 5.3 — Wait for Kafka's answer instead of assuming success.
    committed_positions = consumer.commit(message=message, asynchronous=False)
    if not committed_positions:
        raise RuntimeError("Kafka returned no committed position")

    # Stage 5.4 — Surface an error attached to any returned partition position.
    for position in committed_positions:
        if position.error is not None:
            raise RuntimeError(f"Kafka commit failed: {position.error}")


### Code explanation — every meaningful line in the Stage 5 helpers

| Code | Meaning |
|---|---|
| `def create_order_consumer() -> Consumer:` | Defines a small construction function and promises a Kafka consumer on success. |
| Its docstring | Documents that creation uses the visible configuration, may fail, and should use managed secrets in a real deployment. |
| `if not KAFKA_LIBRARY_AVAILABLE or Consumer is None:` | Checks both the readable flag and the imported class. `or` means either missing condition blocks live execution. |
| `raise RuntimeError(...)` | Gives an actionable installation message. |
| `return Consumer(CONSUMER_CONFIG)` | Creates the client with Stage 1 settings. Construction is not the same as subscription, polling, or processing. |
| `def commit_processed_message(...) -> None:` | Defines the one place that saves progress after business success. |
| Its docstring | Defines the active consumer and processed message inputs, success contract, failure behavior, and throughput tradeoff. |
| `consumer.commit(message=message, asynchronous=False)` | Requests a commit for the position immediately after this message and waits for the broker response. If the record offset is `8`, the next position is normally `9`. |
| `if not committed_positions:` | Treats an absent result as failure rather than pretending progress was saved. |
| `for position in committed_positions:` | Examines every topic-partition position returned by the client. |
| `if position.error is not None:` | Checks whether Kafka attached a failure to that position. |
| `raise RuntimeError(f"...")` | Stops and includes the real commit error in the message. |

Why save offset `9` after processing record `8`? The committed number answers “where should this group read next?” It is a bookmark between records, not a stamp placed on a record.


## Stage 5.5 — The complete polling loop, first in plain English

Before reading the code, read its algorithm:

1. Reject impossible notebook limits.
2. Create one consumer.
3. Subscribe it to the order topic.
4. Ask for a message with `poll()`.
5. If the result is `None`, count the empty wait and ask again.
6. If the message contains an error, stop and report it.
7. Otherwise decode the bytes.
8. Run the business handler.
9. Only after the handler succeeds, commit the next position.
10. Repeat until the notebook limit is reached.
11. Close the consumer whether the loop succeeds or fails.


In [ ]:
def consume_orders(max_messages: int = 5, max_empty_polls: int = 10) -> int:
    """Poll, process, and commit a bounded number of order events.

    Args:
        max_messages: Stop after this many successful process-and-commit cycles.
        max_empty_polls: Stop after this many consecutive one-second empty waits.

    Returns:
        The number of records both processed and committed during this run.

    Raises:
        ValueError: Either notebook limit is not positive.
        InvalidOrderEvent: A record does not match the expected JSON contract.
        KafkaException: Polling or committing fails at the Kafka client boundary.
        Exception: The business handler fails; its original error is preserved.

    Production tradeoff:
        Bounds prevent an accidental endless notebook cell. A long-running service
        normally polls until a shutdown signal and adds retries, metrics, and alerts.
    """
    # Stage 6.1 — Validate the notebook-only stopping limits.
    if max_messages <= 0 or max_empty_polls <= 0:
        raise ValueError("Both stopping limits must be positive")

    # Stage 6.2 — Create local state before entering the loop.
    consumer = create_order_consumer()
    processed_count = 0
    consecutive_empty_polls = 0

    try:
        # Stage 6.3 — Ask Kafka to assign this group work from the order topic.
        consumer.subscribe([ORDER_TOPIC])

        # Stage 6.4 — Continue while neither notebook stopping limit is reached.
        while processed_count < max_messages and consecutive_empty_polls < max_empty_polls:
            message = consumer.poll(1.0)

            # Stage 6.5 — An empty wait is normal; it is not a business record.
            if message is None:
                consecutive_empty_polls += 1
                print("No record arrived during this one-second poll")
                continue

            # Stage 6.6 — Never send a Kafka error through the JSON decoder.
            if message.error():
                raise KafkaException(message.error())

            consecutive_empty_polls = 0
            print(
                "Received:",
                message.topic(),
                "partition", message.partition(),
                "offset", message.offset(),
            )

            # Stage 6.7 — Convert and validate the untrusted bytes.
            event = decode_order_event(message.value())

            # Stage 6.8 — Perform business work before saving progress.
            handle_order_created(event)

            # Stage 6.9 — Commit only because decoding and handling both succeeded.
            commit_processed_message(consumer, message)
            processed_count += 1
            print("Progress committed; processed count:", processed_count)
    finally:
        # Stage 6.10 — Leave the group and release resources on every exit path.
        consumer.close()
        print("Consumer closed")

    return processed_count


### Code explanation — every meaningful line in the complete loop

#### Function contract and setup

| Code | Meaning |
|---|---|
| `def consume_orders(...):` | Defines a bounded notebook consumer. Defaults mean at most five successful records or ten consecutive empty waits. |
| The docstring | States inputs, output, failure behavior, and why a deployed service needs more operational features. |
| `if max_messages <= 0 or ...` | Rejects limits that could make the loop's behavior nonsensical. |
| `consumer = create_order_consumer()` | Creates the Kafka client. If creation fails, there is no client to close yet. |
| `processed_count = 0` | Counts records whose handler **and commit** succeeded in this run. |
| `consecutive_empty_polls = 0` | Counts empty waits in a row so an idle notebook cell eventually ends. |

#### Subscription and polling

| Code | Meaning |
|---|---|
| `try:` | Starts the protected region. Any exit from this region—success or exception—will execute `finally`. |
| `consumer.subscribe([ORDER_TOPIC])` | Subscribes to a list containing one topic. The group assignment is coordinated as polling occurs. |
| `while processed_count < ... and ...:` | Repeats only while both limits still allow it. `and` means reaching either limit ends the loop. |
| `message = consumer.poll(1.0)` | Lets the client wait up to one second for its next event/record, then stores the answer in `message`. |
| `if message is None:` | Selects the ordinary “nothing arrived during this wait” path. |
| `consecutive_empty_polls += 1` | Adds one to the current empty-wait streak. |
| The empty-poll `print(...)` | Makes idleness visible while learning. A real service would normally use metrics or controlled logging. |
| `continue` | Jumps back to the top of the `while` loop. No decode, handler, or commit occurs for `None`. |

#### Error, record, and metadata paths

| Code | Meaning |
|---|---|
| `if message.error():` | Checks whether this returned message represents a Kafka/client error rather than business data. |
| `raise KafkaException(message.error())` | Stops and preserves Kafka's error details. The bad result is never JSON-decoded or committed. |
| `consecutive_empty_polls = 0` | A real record breaks the empty-wait streak. |
| The multi-line `print(...)` | Displays the record's source topic, partition, and offset. The surrounding parentheses only let one function call span readable lines. |

#### Decode, process, commit, and cleanup

| Code | Meaning |
|---|---|
| `event = decode_order_event(message.value())` | Takes raw record bytes, converts them, validates them, and stores the accepted dictionary. Failure stops before business work and commit. |
| `handle_order_created(event)` | Runs the business action. Failure stops before commit. |
| `commit_processed_message(consumer, message)` | Saves the next position only after decode and business work returned successfully. |
| `processed_count += 1` | Counts success only after the commit helper also returned successfully. |
| The success `print(...)` | Shows the completed process-and-commit count. |
| `finally:` | Declares cleanup that must run whether earlier lines succeed, raise, or reach a stopping limit. |
| `consumer.close()` | Leaves the group cleanly and releases network/client resources. Closing is cleanup; it is not proof that business processing succeeded. |
| `print("Consumer closed")` | Makes cleanup visible in the notebook. |
| `return processed_count` | Returns the completed count after cleanup. This line is reached only when no exception escaped. |


---
# Stage 7.0 — Run one controlled live example

The consumer needs a record to read. The next function sends exactly one known record with a tiny producer. It exists only to make this consumer lesson runnable; it is not a second producer tutorial.

Keep `RUN_LIVE_KAFKA = False` while reading. To run the live example:

1. Start the learning broker.
2. Install `confluent-kafka`.
3. Change `RUN_LIVE_KAFKA` to `True` in Stage 1.3.
4. Restart the kernel and run all cells.


In [15]:
def publish_sample_order() -> None:
    """Send the sample event so the live consumer has one known record.

    Args:
        None. The function uses Stage 1 settings and the Stage 2 sample event.

    Returns:
        Nothing after Kafka acknowledges the record.

    Raises:
        RuntimeError: The Kafka package is missing or delivery is incomplete.

    Production tradeoff:
        This helper favors clarity over throughput and waits for all delivery work.
        Production producers normally live in their own application component.
    """
    # Stage 7.1 — Check the optional dependency before constructing a producer.
    if not KAFKA_LIBRARY_AVAILABLE or Producer is None:
        raise RuntimeError("Install confluent-kafka before running the live lab")

    # Stage 7.2 — Create a producer and containers for its callback result.
    producer = Producer({"bootstrap.servers": BOOTSTRAP_SERVERS, "acks": "all"})
    acknowledged_offsets: list[int] = []
    delivery_failures: list[str] = []

    def record_delivery(error: object | None, message: Message) -> None:
        """Capture the asynchronous delivery result for this one-record lab."""
        if error is not None:
            delivery_failures.append(str(error))
            return
        acknowledged_offsets.append(message.offset())

    # Stage 7.3 — Queue the same bytes that Stage 3 decoded successfully.
    producer.produce(
        topic=ORDER_TOPIC,
        key=b"order-1001",
        value=wire_bytes,
        on_delivery=record_delivery,
    )

    # Stage 7.4 — Wait for delivery and reject an uncertain result.
    remaining_messages = producer.flush(10.0)
    if remaining_messages != 0 or delivery_failures or not acknowledged_offsets:
        raise RuntimeError(
            "Sample delivery was incomplete: "
            f"remaining={remaining_messages}, failures={delivery_failures}"
        )
    print("Sample order acknowledged at offset:", acknowledged_offsets[0])


### Code explanation — every meaningful line in the one-record producer helper

| Code | Meaning |
|---|---|
| `def publish_sample_order() -> None:` | Defines a helper that sends one known event and returns no data on success. |
| Its docstring | Documents its existing inputs, completion condition, failure behavior, and why it is not a production producer design. |
| The package check | Prevents calling a missing `Producer` class and gives an actionable error. |
| `Producer({...})` | Creates a producer pointed at the same broker. `acks="all"` asks the configured in-sync replicas to acknowledge the write; the local lab has one broker. |
| `acknowledged_offsets = []` | Stores the offset reported by a successful delivery callback. |
| `delivery_failures = []` | Stores delivery error text instead of losing it. |
| `def record_delivery(...)` | Defines the callback that the producer client invokes when it knows the delivery result. |
| Its one-line docstring | States the callback's single purpose. |
| `if error is not None:` | Selects failed delivery. |
| `delivery_failures.append(str(error))` | Saves readable failure information. |
| `return` | Ends the callback so a failed delivery is not also counted as acknowledged. |
| `acknowledged_offsets.append(message.offset())` | Records the broker-assigned offset on successful delivery. |
| `producer.produce(...)` | Queues one record. The topic, key, bytes, and callback are named explicitly. Queueing alone is not yet proof of broker acknowledgement. |
| `remaining_messages = producer.flush(10.0)` | Waits up to ten seconds for queued delivery work and returns how many messages remain unfinished. |
| The three-part `if` | Treats unfinished work, a reported failure, or a missing acknowledgement as failure. |
| `raise RuntimeError(...)` | Reports the remaining count and captured delivery errors. Adjacent strings inside parentheses form one message. |
| The final `print(...)` | Shows the offset Kafka assigned to the acknowledged record. |


In [16]:
# Stage 7.5 — Network access happens only behind the explicit safety switch.
if RUN_LIVE_KAFKA:
    publish_sample_order()
    completed_count = consume_orders(max_messages=1, max_empty_polls=10)
    print("Live process-and-commit count:", completed_count)
else:
    print("Live Kafka skipped. Set RUN_LIVE_KAFKA=True when the broker is ready.")


Live Kafka skipped. Set RUN_LIVE_KAFKA=True when the broker is ready.


### Code explanation — every line in Stage 7.5

| Code | Meaning |
|---|---|
| The Stage 7.5 comment | Marks the only cell path that performs network work. |
| `if RUN_LIVE_KAFKA:` | Runs the indented live branch only when you deliberately changed the safety switch. |
| `publish_sample_order()` | Sends and waits for acknowledgement of one known record. |
| `consume_orders(...)` | Starts the consumer, stopping after one committed record or ten consecutive empty waits. |
| The success `print(...)` | Displays how many records finished both business work and commit. |
| `else:` | Selects the safe offline branch when the switch remains `False`. |
| The skip `print(...)` | Explains why no network activity occurred and what to change deliberately. |

If this group consumed older records in a previous run, its committed position takes priority over `auto.offset.reset="earliest"`. The newly published record is after that position, so the stable group can still continue forward to it.


---
# Stage 8.0 — The commit rule without jargon

The safe order used by this notebook is:

```text
receive offset 8
      |
      v
decode and validate
      |
      v
business work succeeds
      |
      v
commit next position 9
```

Now consider crashes:

| Failure moment | Saved position | What can happen after restart? |
|---|---:|---|
| Before business work | Still `8` | Kafka can return record `8` again. Good: it was not processed. |
| During failed business work | Still `8` | Kafka can return record `8` again. The error needs repair/retry policy. |
| After business work, before commit | Still `8` | Kafka can return record `8` again, so the business action may repeat. |
| After successful commit | `9` | This group normally continues with the next record. |

The third row is why people say **at-least-once processing**: a record is not silently skipped by committing early, but the same business action can occur more than once after a crash.

The practical production response is an **idempotent handler**. That simply means repeating the same event does not create a second unwanted result. For example, a database can enforce one analytics row per `event_id`.

Kafka's offset commit does not by itself make an external database write exactly once.


## Stage 8.1 — What happens in common situations

| Situation | What this notebook does | Why |
|---|---|---|
| `poll()` returns `None` | Polls again until the notebook idle limit. | One empty wait is normal. |
| Kafka returns an error | Raises `KafkaException`, closes, and does not process/commit it. | An error is not JSON business data. |
| JSON is broken or unsupported | Raises `InvalidOrderEvent`, closes, and does not commit. | Silently skipping unknown data can lose work. |
| Business handler raises | Closes and does not commit. | Saving progress would move past unfinished work. |
| Business succeeds but commit fails | Closes and reports failure; replay is possible. | The external work may exist while Kafka's bookmark did not move. |
| Message succeeds and commit succeeds | Counts it, then polls again or exits at the limit. | Both application work and saved progress completed. |

This policy is called **fail closed**: when uncertain, stop instead of pretending success. Its downside is that one permanently bad record can block a partition. Production systems need an explicit retry and quarantine policy; that is a separate lesson, not something to hide inside the first polling loop.


---
# Stage 9.0 — Consumer groups and scaling, slowly

Imagine one topic with three partitions:

```text
P0 ─────────> Consumer A
P1 ─────────> Consumer A
P2 ─────────> Consumer B

Consumer group: beginner-order-analytics-v1
```

Within one traditional consumer group, one partition is assigned to at most one active consumer at a time. Therefore:

| Partitions | Consumers in the same group | Consumers that can have partition work |
|---:|---:|---:|
| 3 | 1 | 1 |
| 3 | 2 | 2 |
| 3 | 3 | 3 |
| 3 | 5 | At most 3; at least 2 are idle |

If an audit application must independently see every order, give it another group ID. If it uses the analytics group ID, it joins the same team and shares the work instead of receiving an independent copy.

When group members join, leave, or stop responding, Kafka may redistribute partition ownership. That redistribution is called a **rebalance**. You only need the word here; advanced rebalance-safe processing belongs in a later notebook.


# Stage 10.0 — Production words translated into questions

| Term | Ask this ordinary question |
|---|---|
| **Consumer lag** | How many records is this group behind Kafka's latest position? |
| **Throughput** | How many records do we finish per second? |
| **Processing latency** | How long does the business handler take for one record? |
| **Poison record** | Is one permanently unusable record repeatedly blocking progress? |
| **Idempotency** | If the same event runs twice, do we still get one correct outcome? |
| **Graceful shutdown** | When the process is asked to stop, can it finish its current safe step and close the consumer? |
| **Rebalance** | Is Kafka redistributing partitions because group membership changed? |

A real service based on this loop still needs:

- authentication and encrypted connections;
- durable, idempotent business storage;
- controlled retry and bad-record quarantine rules;
- metrics and alerts for lag, failures, and handler time;
- an application shutdown signal instead of notebook limits;
- configuration values appropriate for expected handler duration;
- integration testing against a disposable Kafka broker before deployment.

Those requirements do not change the basic flow. They make each stage observable and resilient.


---
# Final recap — say this out loud

1. A producer sends bytes to a topic.
2. A consumer joins a named group and subscribes to that topic.
3. `poll()` may return nothing, an error, or a normal message.
4. A normal message contains bytes plus topic, partition, and offset metadata.
5. Our code converts bytes to text, parses JSON, and validates the event.
6. The business handler performs the useful work.
7. Only after that work succeeds do we commit the group's next position.
8. A crash between business success and commit can cause the event to run again.
9. `close()` releases the consumer; it does not mean business work succeeded.

The one sentence to remember is:

> **Poll gives your program a record. Process does the useful work. Commit saves where the group should continue. These are three different events.**

## Next learning step

Only after this notebook feels obvious, study offset-management failure windows in depth: auto commit versus manual commit, record-by-record versus batch commit, replay, and idempotent processing.

Authoritative references:

- [Confluent Python client — Consumer API](https://docs.confluent.io/platform/current/clients/confluent-kafka-python/html/index.html#consumer)
- [Confluent Python client overview — consuming messages](https://docs.confluent.io/kafka-clients/python/current/overview.html#ak-consumer)
